In [ ]:
import json
import os
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

True

In [3]:
import os
import shutil
import sys
import subprocess

def ensure_tesseract_installed():
    """Ensure Tesseract is installed and available"""
    try:
        import pytesseract
    except ImportError:
        print("📦 Installing pytesseract Python package...")
        subprocess.run([sys.executable, "-m", "pip", "install", "pytesseract"], check=True)
        import pytesseract

    # Check if tesseract is available in PATH
    if shutil.which("tesseract") is not None:
        print("✅ Tesseract is available on PATH")
        return
    
    raise RuntimeError(
        "❌ Tesseract not found in PATH.\n"
        "E:\\tesseract has been added to PATH.\n"
        "Please restart your kernel (Ctrl+Shift+F10) for changes to take effect."
    )

ensure_tesseract_installed()

def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")

    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True,
    )

    print(f"✅ Extracted {len(elements)} elements")
    return elements

# Test with your PDF file
file_path = "./docs/Google.pdf"
elements = partition_document(file_path)

✅ Tesseract is available on PATH
📄 Partitioning document: ./docs/Google.pdf


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
No languages specified, defaulting to English.
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 2654.69it/s]


✅ Extracted 811 elements


In [ ]:
import pytesseract

# Verify tesseract is working (should find it from PATH now)
try:
    result = pytesseract.pytesseract.get_tesseract_version()
    print(f"✅ Tesseract is installed and working!")
    print(f"Version: {result}")
except Exception as e:
    print(f"❌ Tesseract error: {e}")
    print("If the error persists, restart the kernel (Ctrl+Shift+F10)")

❌ Tesseract error: tesseract is not installed or it's not in your PATH. See README file for more information.
Please verify: E:\tesseract\tesseract.exe exists


In [5]:
#elements

# All types of different atomic elements we see from unstructured
set([str(type(el)) for el in elements])
elements[36].to_dict()

{'type': 'UncategorizedText',
 'element_id': '82d783632eddedefb8473de2c532a67e',
 'text': 'Area served',
 'metadata': {'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(1586.1652984931097),
     np.float64(3674.648935590277)),
    (np.float64(1586.1652984931097), np.float64(3725.982266420138)),
    (np.float64(1880.0690726548332), np.float64(3725.982266420138)),
    (np.float64(1880.0690726548332), np.float64(3674.648935590277))),
   'system': 'PixelSpace',
   'layout_width': 2897,
   'layout_height': 4093},
  'last_modified': '2026-08-04T16:45:11',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'file_directory': './docs',
  'filename': 'Google.pdf'}}

In [6]:
# Gather all images
images = [element for element in elements if element.category == 'Image']
print(f"Found {len(images)} images")

images[0].to_dict()


Found 28 images


{'type': 'Image',
 'element_id': 'f2ff962c1ba150a8580089fe3ff48841',
 'text': '',
 'metadata': {'coordinates': {'points': ((np.float64(1611.458266189236),
     np.float64(1159.374951692708)),
    (np.float64(1611.458266189236), np.float64(1906.7707538845484)),
    (np.float64(2632.2916269878438), np.float64(1906.7707538845484)),
    (np.float64(2632.2916269878438), np.float64(1159.374951692708))),
   'system': 'PixelSpace',
   'layout_width': 2897,
   'layout_height': 4093},
  'last_modified': '2026-08-04T16:45:11',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': './docs',
  'filename': 'Google.pdf'}}

In [7]:
# Gather all table
tables = [element for element in elements if element.category == 'Table']
print(f"Found {len(tables)} tables")

tables[0].to_dict()

Found 6 tables


{'type': 'Table',
 'element_id': '1eca5fa37d333f3ebd6d1902f6216825',
 'text': 'FY in million USD [218][219] References 1999 0.22 −6.0 [220] 2000 19.1 −14.6 [221] 2001 86.4 6.9 284 [221] 2002 439 99.6 682 [221] in billion USD 2003 1.4 0.10 1,628 [221] 2004 3.1 0.39 3,021 [221] 2005 6.1 1.4 5,680 2006 10.6 3.0 10,674 2007 16.5 4.2 16,805 2008 21.8 4.2 20,222 2009 23.6 6.5 19,835 2010 29,3 8.5 24,400 2011 37.9 9.7 32,467 2012 46.0 10.7 53,861 2013 55.5 12.7 47,756',
 'metadata': {'detection_class_prob': 0.8913231492042542,
  'is_extracted': 'partial',
  'coordinates': {'points': ((np.float64(206.13204956054688),
     np.float64(215.80726623535156)),
    (np.float64(206.13204956054688), np.float64(2025.514892578125)),
    (np.float64(1614.7294921875), np.float64(2025.514892578125)),
    (np.float64(1614.7294921875), np.float64(215.80726623535156))),
   'system': 'PixelSpace',
   'layout_width': 2897,
   'layout_height': 4093},
  'last_modified': '2026-08-04T16:45:11',
  'text_as_html': '<t

In [8]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        elements, # The parsed PDF elements from previous step
        max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2400, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# Create chunks
chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 122 chunks


In [11]:
# View all chunks
#chunks

# All unique types
set([str(type(chunk)) for chunk in chunks])

{"<class 'unstructured.documents.elements.CompositeElement'>",
 "<class 'unstructured.documents.elements.Table'>"}

In [14]:
# View a single chunk
chunks[0].to_dict()

# View original elements
#chunks[11].metadata.orig_elements[-1].to_dict()
# Note: 4th chunk has the first image + 11th chunk has the first table in the sample PDF

{'type': 'CompositeElement',
 'element_id': 'ee71bf47-2b8e-4070-903b-4210fe223ff3',
 'text': 'WIKIPEDIA\n\n25 years of the free encyclopedia\n\nGoogle\n\nⓘ\n\nGoogle LLC (/ˈɡuː.ɡəl/ , GOO-gəl) is an American multinational technology corporation focused on information technology, online advertising, search engine technology, email, cloud computing, software, quantum computing, e-commerce, consumer electronics, and artificial intelligence (AI).[9] It has been referred to as "the most powerful company in the world" by the BBC,[10] and is one of the world\'s most valuable brands.[11][12][13] Google\'s parent company Alphabet Inc. has been described as a Big Tech company.\n\nGoogle was founded in 1998 by American computer scientists Larry Page and Sergey Brin. Together, they own about 14% of its publicly listed shares and control 56% of its stockholder voting power through super- voting stock. The company went public via an initial public offering (IPO) in 2004. In 2015, Google was reorgani

In [ ]:
def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content using Meta Muse Glimmer."""
    
    try:
        api_key = os.getenv("NVIDIA_API_KEY")
        if not api_key:
            raise ValueError("Set NVIDIA_API_KEY in your environment before running the enhanced summary.")

        # Initialize OpenAI-compatible client for NVIDIA Integrate API
        client = OpenAI(
            base_url="https://integrate.api.nvidia.com/v1",
            api_key=api_key,
        )

        prompt_text = "You are creating a searchable description for document content retrieval.\n\n"
        prompt_text += "CONTENT TO ANALYZE:\n"
        prompt_text += f"TEXT CONTENT:\n{text}\n\n"

        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"

        if images:
            prompt_text += "IMAGES ARE PROVIDED IN THE MESSAGE AS VISUAL INPUT.\n"

        prompt_text += """
        YOUR TASK:
        Generate a comprehensive, searchable description that covers:

        1. Key facts, numbers, and data points from text and tables
        2. Main topics and concepts discussed
        3. Questions this content could answer
        4. Visual content analysis (charts, diagrams, patterns in images)
        5. Alternative search terms users might use

        Make it detailed and searchable - prioritize findability over brevity.

        SEARCHABLE DESCRIPTION:
        """

        message_content = [{"role": "user", "content": [{"type": "text", "text": prompt_text}]}]

        for image_base64 in images:
            message_content[0]["content"].append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"},
            })

        response = client.chat.completions.create(
            model="meta/muse-glimmer-30b",
            messages=message_content,
            temperature=1,
            top_p=0.95,
            max_tokens=8192,
            stream=False,
        )

        return response.choices[0].message.content

    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary
